In [1]:
import polars as pl
import torch
from transformers import pipeline
from tqdm import tqdm
import os

# --- CONFIG ---
batch_size = 256 
parquet_path = "../../data/raw/item_list.parquet"
output_path = "../../data/processed/item_list_sentiment.parquet"

# Create output folder if it does not yet exist
os.makedirs("../../data/processed", exist_ok=True)

# --- Read Data ---
item_list = pl.read_parquet(parquet_path)
items = item_list["item"].to_list()
print(f"Processing {len(items)} items...")

# --- Define Pipelines ---
# Define pipeline configuration
pipe_kwargs = {
    "device": 0, 
    "batch_size": batch_size, 
    "model_kwargs": {"torch_dtype": torch.float16}
}

# Define Pipelines using 'top_k=None' to get full probability vectors
print("Initializing Pipelines...")
# Model A: Sentiment (Negative, Neutral, Positive)
sent_pipe = pipeline("text-classification", 
                     model="cardiffnlp/twitter-roberta-base-sentiment-latest", 
                     top_k=None, **pipe_kwargs)

# Model B: 7-Emotion Vector (Anger, Disgust, Fear, Joy, Neutral, Sadness, Surprise)
hart_pipe = pipeline("text-classification", 
                     model="j-hartmann/emotion-english-distilroberta-base", 
                     top_k=None, **pipe_kwargs)

# --- Execute Pipelines ---
print("Running Sentiment...")
sent_res = sent_pipe(items, truncation=True, max_length=64)

print("Running Hartmann Emotion Vector...")
hart_res = hart_pipe(items, truncation=True, max_length=64)

# --- Extract Scores into Columns ---
# Cardiff Sentiment [Negative, Neutral, Positive]
sent_cols = {f"sent_{label}": [] for label in [r['label'] for r in sent_res[0]]}
for res in sent_res:
    for r in res:
        sent_cols[f"sent_{r['label']}"].append(r['score'])

# Hartmann [7 emotions]
hart_cols = {f"emo_{label}": [] for label in [r['label'] for r in hart_res[0]]}
for res in hart_res:
    for r in res:
        hart_cols[f"emo_{r['label']}"].append(r['score'])

# --- Build Final Table ---
# Create Polars dataframe using the score vectors defined before
new_features = pl.DataFrame({**sent_cols, **hart_cols})

# Adding metadata columns (top labels for quick inspection)
new_features = new_features.with_columns([
    pl.Series("top_sentiment", [r[0]['label'] for r in sent_res]), # First in sorted list
    pl.Series("top_emotion", [r[0]['label'] for r in hart_res])
])

# Bind the the new features into the item list
final_df = pl.concat([item_list, new_features], how="horizontal")

# --- Save data to disk ---
final_df.write_parquet(output_path)
print(f"Success! Saved {len(final_df)} items to {output_path}")

c:\Users\alexm\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing 2027 items...
Initializing Pipelines...


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
Device set to use cuda:0


Running Sentiment...
Running Hartmann Emotion Vector...
Success! Saved 2027 items to ../../data/processed/item_list_sentiment.parquet
